# Loading

In [ ]:
from plot_grn import create_network_from_dict,grn_network_from_dict
import omicverse as ov
import pandas as pd
import json
import numpy as np
from adjustText import adjust_text
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
ov.plot_set(font_path='Arial')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

In [ ]:
def split_long_name(name, threshold=50):
    if len(name) > threshold:
        middle = len(name) // 2
        split_point = middle
        while split_point > 0 and name[split_point] != ' ':
            split_point -= 1
        if split_point > 0:  
            return name[:split_point] + '\n' + name[split_point+1:]
        else:
            return name
    else:
        return name
        
def pathway_enrichment(enr,
                      figsize=(6, 6),
                      color='#aec7e8',
                      threshold=50,fontsize=14,
                      linewidth=1.5,
                      title=''):
    """
    Plot the enrichment results with split long names.

    Parameters:
        enr: DataFrame containing the enrichment results with columns 'Term' and 'Adjusted P-value'.
        figsize: Tuple specifying the figure size (width, height).
        color: String specifying the color of the bars.
        threshold: Integer specifying the maximum length of a term before it is split.
        linewidth: Float specifying the line width of the horizontal line.
        title: String specifying the title of the plot.
    """
    enr['Term'] = enr['Term'].str.replace('\(GO:\d+\)', '', regex=True)
    enr['-log10(Adjusted P-value)'] = -np.log10(enr['Adjusted P-value'])
    enr_sorted = enr.sort_values('-log10(Adjusted P-value)', ascending=False)
    enr_sorted['Term'] = enr_sorted['Term'].apply(lambda x: split_long_name(x, threshold))

    f, ax = plt.subplots(1, 1, figsize=figsize, sharex=True) 
    colors = [color] * len(enr.index)
    barplot = sns.barplot(x="-log10(Adjusted P-value)",
                          y="Term", data=enr_sorted, palette=colors, ax=ax)


    for i, p in enumerate(ax.patches):  
        ax.text(p.get_x()+0.1, p.get_y() + p.get_height() / 2. + 0.05,
                f'{enr_sorted["Term"].iloc[i]}', fontsize=fontsize,
                ha='left', va='center', color='black')

    ax.set_title(title,fontweight='bold',fontsize=fontsize+1)
    ax.set_ylabel('')

    ax.spines['bottom'].set_visible(True)
    ax.spines['bottom'].set_linewidth(linewidth)
    ax.spines['left'].set_visible(True)
    ax.spines['left'].set_linewidth(linewidth)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    ax.set_xlabel('-log$_{10}$ Adjusted $\it{P}$',fontsize=16,)
    plt.setp(ax.get_yticklabels(), visible=False)
    plt.tight_layout(h_pad=2)
    plt.grid(False)

    return f,ax

def top_n_unique_genes(pathway_to_genes, df, gene_col="gene", expr_col="mean_counts",n=5):
    all_genes = []
    for pw, genes in pathway_to_genes.items():
        tmp = df[df[gene_col].isin(genes)].sort_values(expr_col, ascending=False)
        all_genes.extend(tmp[gene_col].head(8).tolist())
    seen = set()
    unique_genes = []
    for g in all_genes:
        if g not in seen:
            seen.add(g)
            unique_genes.append(g)
    return unique_genes

In [ ]:
pathway_dict=ov.utils.geneset_prepare('genesets/GO_Biological_Process_2021.txt',organism='Human')

In [ ]:
adata = sc.read_h5ad("Process_Data/bin100_3D_region/combined_adata_HIP.h5ad")
sc.pp.calculate_qc_metrics(adata, qc_vars=[], inplace=True, log1p=True)
adata_var = adata.var
adata_var['gene'] = adata_var.index
del adata

In [ ]:
with open('Output/GRN_GPU_output/HIP/tmp_files/regulons.json', 'r', encoding='utf-8') as file:
    grn_dict = json.load(file)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import omicverse as ov 

def run_grn_analysis_plot(keys, grn_dict, pathway_dict, adata_var, save_prefix, other_target_size=10,specific_target_size=150,threshold=30, ):
    # 确保保存目录存在
    save_dir = os.path.dirname(save_prefix)
    if save_dir and not os.path.exists(save_dir):
        os.makedirs(save_dir)

    # 1. 数据准备
    grn_dict_sub = {k: grn_dict[k] for k in keys if k in grn_dict}
    s = pd.Series(grn_dict_sub)
    df = s.explode().reset_index()
    df.columns = ['TF', 'Target Gene']

    # 2. 通路富集分析
    enr = ov.bulk.geneset_enrichment(gene_list=list(set(df['Target Gene'])),
                                     pathways_dict=pathway_dict,
                                     pvalue_type='auto',
                                     organism='Human')

    pathway_list = enr.sort_values(['Adjusted P-value'])['Term'][:5].tolist()
    subset = enr[enr['Term'].isin(pathway_list)]

    pathway_to_genes = {}
    for index, row in subset.iterrows():
        pathway_to_genes[row['Term']] = [g.strip() for g in row['Genes'].split(';') if g.strip()]

    enr = enr[enr['Term'].isin(pathway_to_genes.keys())]

    # 3. 绘制并保存富集图
    fig, ax = pathway_enrichment(enr.sort_values(['Adjusted P-value'], ascending=True).iloc[0:5, :],
                                 color='#B0D8E8', fontsize=15, figsize=(6, 4), threshold=threshold, title='')
    ax.set_xlabel('-log$_{10}$ Adjusted $\it{P}$', fontsize=16)
    plt.grid(False)
    ax.tick_params(axis='x', which='major', labelsize=16, width=1.5, length=4, direction='out')
    ax.tick_params(axis='y', which='major', labelsize=16, width=1.5, length=0, direction='out')
    
    # 保存路径参考：Figure/HIP/HIP_sc19_selected_GRN_pathway.pdf
    fig.savefig(f"{save_prefix}_selected_GRN_pathway.pdf", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig) # 关闭画布防止重叠

    # 4. 绘制并保存网络图
    G, G_type, G_color = create_network_from_dict(grn_dict_sub, n_top=5000, 
                                                  color_dict={'tf_color': '#7030a0', 'target_color': '#ffc000'})
    
    fig, ax = grn_network_from_dict(
        G, G_type, G_color, grn_dict_sub,
        figsize=(8, 6.5), pos_scale=7, TF_size=300,
        node_alpha=0.8, node_linewidths=0.5,
        TF_fontsize=12, target_fontsize=11,
        specific_gene=top_n_unique_genes(pathway_to_genes, adata_var),
        specific_target_size=specific_target_size,
        other_target_size=other_target_size
    )
    
    # 自动从 save_prefix 提取标题名 (例如从 Figure/HIP/HIP_sc20 提取 HIP_sc20)
    title_name = os.path.basename(save_prefix)
    plt.title(f'{title_name} GRN', fontsize=15, weight='bold')
    
    # 保存路径参考：Figure/HIP/HIP_sc20_GRN.pdf
    fig.savefig(f"{save_prefix}_GRN.pdf", dpi=300, bbox_inches="tight")
    plt.show()

# sc_18

In [ ]:
keys = ['TCF4(+)','NFIB(+)','JUN(+)']

# 直接传入路径前缀，函数会自动补全后缀
run_grn_analysis_plot(
    keys=keys,
    grn_dict=grn_dict,
    pathway_dict=pathway_dict,
    adata_var=adata_var,
    save_prefix="Figure/HIP/HIP_sc18",  # 这里修改你的具体路径前缀
    threshold=50,
    other_target_size=10,
    specific_target_size=100,
)

# sc_19

In [ ]:
keys = ['FOSL2(+)','NFIB(+)','BACH2(+)']

# 直接传入路径前缀，函数会自动补全后缀
run_grn_analysis_plot(
    keys=keys,
    grn_dict=grn_dict,
    pathway_dict=pathway_dict,
    adata_var=adata_var,
    save_prefix="Figure/HIP/HIP_sc18",  # 这里修改你的具体路径前缀
    threshold=50,
    other_target_size=20,
    specific_target_size=150,
)

# sc_20

In [ ]:
keys = ['TCF4(+)','SOX11(+)','NFIB(+)']

# 直接传入路径前缀，函数会自动补全后缀
run_grn_analysis_plot(
    keys=keys,
    grn_dict=grn_dict,
    pathway_dict=pathway_dict,
    adata_var=adata_var,
    save_prefix="Figure/HIP/HIP_sc20"  # 这里修改你的具体路径前缀
)

# sc_21

In [ ]:
keys = ['TCF4(+)','NFIA(+)','HES6(+)']

# 直接传入路径前缀，函数会自动补全后缀
run_grn_analysis_plot(
    keys=keys,
    grn_dict=grn_dict,
    pathway_dict=pathway_dict,
    adata_var=adata_var,
    save_prefix="Figure/HIP/HIP_sc21",  # 这里修改你的具体路径前缀
    threshold=50,
    other_target_size=5,
    specific_target_size=75,
)

# sc_22

In [ ]:
keys = ['TCF4(+)','NFIA(+)','SOX11(+)']

# 直接传入路径前缀，函数会自动补全后缀
run_grn_analysis_plot(
    keys=keys,
    grn_dict=grn_dict,
    pathway_dict=pathway_dict,
    adata_var=adata_var,
    save_prefix="Figure/HIP/HIP_sc22",  # 这里修改你的具体路径前缀
    threshold=50,
    other_target_size=5,
    specific_target_size=75,
)

# sc_23

In [ ]:
keys = ['TCF4(+)','SOX9(+)','EMX2(+)','LEF1(+)']

# 直接传入路径前缀，函数会自动补全后缀
run_grn_analysis_plot(
    keys=keys,
    grn_dict=grn_dict,
    pathway_dict=pathway_dict,
    adata_var=adata_var,
    save_prefix="Figure/HIP/HIP_sc23",  # 这里修改你的具体路径前缀
    threshold=50,
    other_target_size=10,
    specific_target_size=100,
)